# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://mlcommons.org/croissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We print a summary of record sets, each field, and their `@id` identifiers.

In [ ]:
# Print all available record sets and their fields
record_sets = dataset.metadata.record_sets
for record_set in record_sets:
    print(f"RecordSet: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (id: {field.id}, data_type: {field.data_type})")
    print("")

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames for analysis. Use the record set and field `@id`s from the overview above.

For demonstration, the first record set will be used.

In [ ]:
# Extract data from each record set
dataframes = {}

record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"RecordSet: {record_set_id} => Columns: {df.columns.tolist()}")
    print(df.head(2))
    print("")
# Select a record set for further analysis
selected_record_set_id = record_set_ids[0] if record_set_ids else None
if selected_record_set_id:
    print(f"Using record set: {selected_record_set_id}")
    print(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming distributions, or grouping data by key attributes.

Below, we select the first numeric field found for demonstration.

In [ ]:
import numpy as np

if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id]
    # Find a numeric field (schema:Float or schema:Integer)
    first_numeric = None
    group_field = None
    fields = [f for f in dataset.metadata.record_sets if f.id == selected_record_set_id][0].fields
    for f in fields:
        if f.data_type in ('schema:Float','schema:Integer'):
            first_numeric = f.id
            break
    # Choose a group field (categorical, string-type, not numeric)
    for f in fields:
        if f.data_type in ('schema:Text',) and f.id != first_numeric:
            group_field = f.id
            break

    if first_numeric and first_numeric in df.columns:
        try:
            df[first_numeric] = pd.to_numeric(df[first_numeric], errors='coerce')
            threshold = df[first_numeric].mean() if not np.isnan(df[first_numeric].mean()) else 0
            filtered_df = df[df[first_numeric] > threshold]
            print(f"Filtered records with {first_numeric} > {threshold:.2f} : {filtered_df.shape[0]} rows")
            # Normalize
            filtered_df = filtered_df.copy()
            filtered_df[f"{first_numeric}_normalized"] = (
                filtered_df[first_numeric] - filtered_df[first_numeric].mean()
            ) / (filtered_df[first_numeric].std() if filtered_df[first_numeric].std() else 1)
            print(filtered_df[[first_numeric, f"{first_numeric}_normalized"]].head())

            if group_field and group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[first_numeric].mean().reset_index()
                print(f"Grouped data by {group_field} (mean of {first_numeric}):")
                print(grouped_df.head())
        except Exception as e:
            print(f"Could not process numeric analysis: {e}")
    else:
        print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we plot the distribution and (optionally) group means for the selected numeric field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id is not None and first_numeric and first_numeric in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[first_numeric].dropna(), bins=40, kde=True)
    plt.title(f"Distribution of {first_numeric}")
    plt.xlabel(first_numeric)
    plt.ylabel("Count")
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,4))
        order = df[group_field].value_counts().index[:10]
        sns.boxplot(data=df, x=group_field, y=first_numeric, order=order)
        plt.title(f"{first_numeric} grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(first_numeric)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded and reviewed the dataset metadata and structure using `mlcroissant`.
- Identified available record sets and field identifiers (`@id`s).
- Loaded data into pandas DataFrames for each record set by referencing their `@id`s.
- Performed basic EDA, including filtering and normalization on selected numeric fields by their `@id`.
- Visualized distributions and grouped summaries where possible.

This exploration demonstrates how structured metadata using `mlcroissant` can facilitate programmatic analysis across semantically rich datasets.